In [7]:
import requests
from pathlib import Path
from tqdm import tqdm

In [ ]:
BASE_URL = "https://dataverse.nl/api"
DOI = "doi:10.34894/AECRSD"

ALLOWED_PREFIXES = ("training/", "test/")

# Project path for dataset
PROJECT_ROOT = Path("..").resolve()
OUT_ROOT = PROJECT_ROOT / "data" / "raw" / "WMH"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Downloading to:", OUT_ROOT)

In [9]:
# Fetch dataset metadata
print("Fetching dataset metadata...")
r = requests.get(
    f"{BASE_URL}/datasets/:persistentId/versions/:latest",
    params={"persistentId": DOI},
    timeout=60,
)
r.raise_for_status()

files = r.json()["data"]["files"]
print(f"Found {len(files)} total files in dataset")

# Filter training + test
selected = []
for f in files:
    directory = f.get("directoryLabel", "")
    if directory.startswith(ALLOWED_PREFIXES):
        selected.append(f)

print(f"Will download {len(selected)} files into {OUT_ROOT}/")

Fetching dataset metadata...
Found 1791 total files in dataset
Will download 1670 files into /Users/nicholasborch/Desktop/UNI/Bachelor Project/Repo/Bachelor-Project/data/raw/MICCAI17/


In [10]:
# Download files
session = requests.Session()

for f in tqdm(selected, desc="Downloading WMH files"):
    directory = f.get("directoryLabel", "")      
    filename = f["dataFile"]["filename"]   
    file_id = f["dataFile"]["id"]

    local_dir = OUT_ROOT / directory
    local_dir.mkdir(parents=True, exist_ok=True)

    local_path = local_dir / filename

    # Skip already-downloaded files
    if local_path.exists():
        continue

    url = f"{BASE_URL}/access/datafile/{file_id}"
    with session.get(url, stream=True, timeout=120) as rf:
        rf.raise_for_status()
        with open(local_path, "wb") as out:
            for chunk in rf.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    out.write(chunk)

In [11]:
# Sanity check
for split in ["training", "test"]:
    split_path = OUT_ROOT / split
    print(split, "exists:", split_path.exists(), "| path:", split_path)

training exists: True | path: /Users/nicholasborch/Desktop/UNI/Bachelor Project/Repo/Bachelor-Project/data/raw/MICCAI17/training
test exists: True | path: /Users/nicholasborch/Desktop/UNI/Bachelor Project/Repo/Bachelor-Project/data/raw/MICCAI17/test
